# Hierarchical RAG with AWS

## 📖 Overview

This notebook demonstrates **Hierarchical RAG** using AWS services:
- **Parent-child document relationships** for context preservation
- **Small chunks for retrieval** (precision)
- **Large chunks for generation** (context)
- **Automatic parent fetching** when child matched

### What is Hierarchical RAG?

**Flat RAG (Standard):**
```
Document → Split into Chunks → Index
Query → Find Chunk → Use Chunk
│
└─ Problem: Small chunks lack context!
```

**Hierarchical RAG:**
```
Document → Parent Chunks (large)
              ↓
           Child Chunks (small)
              ↓
Query → Find Child (precise) → Fetch Parent (context)
```

### Architecture

```mermaid
graph TB
    A[Document] --> B[Split into<br/>Parent Chunks]
    B --> C1[Parent 1<br/>500 tokens]
    B --> C2[Parent 2<br/>500 tokens]
    C1 --> D1[Child 1a<br/>100 tokens]
    C1 --> D2[Child 1b<br/>100 tokens]
    C2 --> D3[Child 2a<br/>100 tokens]
    E[User Query] --> F[Search Children<br/>OpenSearch]
    F --> G[Match: Child 1b]
    G --> H[Fetch Parent:<br/>Parent 1]
    H --> I[Generate with<br/>Parent Context]
    I --> J[Answer with<br/>Full Context]
    style G fill:#f3e5f5
    style H fill:#ffe0b2
    style J fill:#b3e5fc
```

## 1️⃣ Setup & Imports

In [ ]:
import sys
import json
from typing import List, Dict
import time
from dataclasses import dataclass
import uuid

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

## 2️⃣ Configuration

In [ ]:
AWS_REGION = 'us-west-2'
INDEX_NAME = 'hierarchical_rag_docs'
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
LLM_MODEL = 'us.anthropic.claude-sonnet-4-6'
PARENT_CHUNK_SIZE = 500
CHILD_CHUNK_SIZE = 100
RETRIEVAL_TOP_K = 3

print(f"Configuration:")
print(f"  Parent chunks: {PARENT_CHUNK_SIZE} tokens")
print(f"  Child chunks: {CHILD_CHUNK_SIZE} tokens")

## 3️⃣ Initialize Services

In [ ]:
opensearch = OpenSearchManager(region=AWS_REGION, index_name=INDEX_NAME)
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
llm = BedrockLLM(AWS_REGION, LLM_MODEL, temperature=0.7)

print("✓ Services initialized")

## 4️⃣ Document Hierarchy

In [ ]:
@dataclass
class ParentChunk:
    parent_id: str
    text: str
    embedding: List[float]
    metadata: Dict

@dataclass
class ChildChunk:
    child_id: str
    parent_id: str
    text: str
    embedding: List[float]
    metadata: Dict

def create_hierarchy(document: str, parent_size: int = 500, child_size: int = 100) -> tuple:
    words = document.split()
    parent_words = int(parent_size / 1.3)
    child_words = int(child_size / 1.3)
    parents, children = [], []
    
    for i in range(0, len(words), parent_words):
        parent_text = ' '.join(words[i:i + parent_words])
        parent_id = str(uuid.uuid4())
        parent_emb = embedder.embed_text(parent_text)
        
        parent = ParentChunk(
            parent_id=parent_id,
            text=parent_text,
            embedding=parent_emb,
            metadata={'chunk_type': 'parent'}
        )
        parents.append(parent)
        
        parent_words_list = parent_text.split()
        for j in range(0, len(parent_words_list), child_words):
            child_text = ' '.join(parent_words_list[j:j + child_words])
            child_id = str(uuid.uuid4())
            child_emb = embedder.embed_text(child_text)
            
            child = ChildChunk(
                child_id=child_id,
                parent_id=parent_id,
                text=child_text,
                embedding=child_emb,
                metadata={'chunk_type': 'child', 'parent_id': parent_id}
            )
            children.append(child)
    
    return parents, children

print("✓ Hierarchy framework ready")

## 5️⃣ Create Hierarchical Index

In [ ]:
sample_document = """
AWS Bedrock is a fully managed service that provides API access to high-performing foundation models from leading AI companies. 
Through a single API, you can choose from various models including Claude from Anthropic, Titan from Amazon, and others. 
Bedrock offers serverless access, meaning you don't need to manage any infrastructure.

Claude Opus is the most capable model in the Claude family, designed for complex reasoning tasks. 
It costs $15 per million input tokens and $75 per million output tokens. Opus excels at deep understanding and analysis.

Claude Sonnet provides balance between quality and cost at $3 input and $15 output per million tokens. 
Sonnet is suitable for most general-purpose tasks including content creation and data extraction.

Claude Haiku is optimized for speed and cost efficiency, priced at $0.25 input and $1.25 output per million tokens. 
It's 20 times cheaper than Opus while maintaining good quality for straightforward tasks.

When choosing between models, consider your requirements. Use Opus for complex analysis, Sonnet for balanced tasks, 
and Haiku for high-volume cost-sensitive applications.

OpenSearch Serverless complements Bedrock for RAG applications by providing vector search without cluster management. 
It uses HNSW algorithm for fast similarity search.
"""

print("Creating hierarchy...")
parents, children = create_hierarchy(sample_document, PARENT_CHUNK_SIZE, CHILD_CHUNK_SIZE)
print(f"  Parents: {len(parents)}, Children: {len(children)}")

parent_map = {p.parent_id: p for p in parents}

opensearch.create_index(embedding_dim=embedder.get_embedding_dimension(), force_recreate=True)

child_documents = [{
    'text': child.text,
    'embedding': child.embedding,
    'metadata': {'child_id': child.child_id, 'parent_id': child.parent_id, 'chunk_type': 'child'}
} for child in children]

opensearch.index_documents(child_documents)
print(f"  Indexed {len(child_documents)} children")
print("✓ Hierarchical index created")

## 6️⃣ Hierarchical RAG Pipeline

In [ ]:
def hierarchical_rag(query: str, top_k: int = 3) -> Dict:
    start_time = time.time()
    print(f"Query: {query}\n")
    
    print("Step 1: Search children")
    query_emb = embedder.embed_text(query)
    child_results = opensearch.vector_search(query_emb, top_k=top_k)
    print(f"  Found {len(child_results)} children")
    
    print("\nStep 2: Fetch parents")
    parent_ids = set(r['metadata'].get('parent_id') for r in child_results if r['metadata'].get('parent_id'))
    parent_contexts = [parent_map[pid].text for pid in parent_ids if pid in parent_map]
    print(f"  Retrieved {len(parent_contexts)} parents")
    
    print("\nStep 3: Generate")
    answer = llm.generate_with_context(query, parent_contexts) if parent_contexts else "No information found"
    
    total_time = time.time() - start_time
    print(f"  Completed in {total_time:.2f}s\n")
    
    return {
        'answer': answer,
        'matched_children': [r['text'] for r in child_results],
        'parent_contexts': parent_contexts,
        'metadata': {
            'total_time': total_time,
            'num_children': len(child_results),
            'num_parents': len(parent_contexts),
            'context_size': sum(len(p) for p in parent_contexts)
        }
    }

print("✓ Pipeline ready")

## 7️⃣ Test Hierarchical RAG

In [ ]:
test_query = "What is Claude Haiku optimized for?"
result = hierarchical_rag(test_query, top_k=RETRIEVAL_TOP_K)

print("=" * 70)
print("HIERARCHY IN ACTION")
print("=" * 70)
print(f"\nMatched Children:")
for i, child in enumerate(result['matched_children'], 1):
    print(f"  {i}. {child[:80]}...")

print(f"\nParent Contexts:")
for i, parent in enumerate(result['parent_contexts'], 1):
    print(f"  {i}. {len(parent)} chars")

print(f"\nAnswer:\n{result['answer'][:300]}...")
print(f"\nContext gain: {result['metadata']['context_size'] / sum(len(c) for c in result['matched_children']):.1f}x")

## 8️⃣ Cost Analysis

In [ ]:
evaluator = RAGEvaluator()

print("=" * 70)
print("COST ANALYSIS")
print("=" * 70)

query_cost = evaluator.estimate_cost(
    num_queries=1,
    num_docs=0,
    avg_doc_length=650,
    model_name='opus'
)

print(f"\nPer Query:")
print(f"  Search: {len(children)} child embeddings")
print(f"  Generation: ~2 parent chunks")
print(f"  Cost: ${query_cost['total_cost']:.4f}")

simple_rag_cost = 0.08
overhead = ((query_cost['total_cost'] - simple_rag_cost) / simple_rag_cost) * 100

print(f"\nComparison:")
print(f"  Simple RAG: ${simple_rag_cost:.4f}")
print(f"  Hierarchical: ${query_cost['total_cost']:.4f}")
print(f"  Overhead: {overhead:+.0f}%")
print(f"\n  Trade-off: {overhead:+.0f}% cost for 3-5x more context")

## 9️⃣ Performance Metrics

In [ ]:
print("=" * 70)
print("PERFORMANCE METRICS")
print("=" * 70)

test_set = [
    "What is AWS Bedrock?",
    "Compare Claude Opus and Haiku pricing",
    "How does OpenSearch help with RAG?"
]

metrics = {'total_time': 0, 'context_gains': []}

for query in test_set:
    result = hierarchical_rag(query, top_k=3)
    metrics['total_time'] += result['metadata']['total_time']
    child_size = sum(len(c) for c in result['matched_children'])
    parent_size = result['metadata']['context_size']
    metrics['context_gains'].append(parent_size / child_size if child_size > 0 else 0)

avg_time = metrics['total_time'] / len(test_set)
avg_gain = sum(metrics['context_gains']) / len(metrics['context_gains'])

print(f"\nResults ({len(test_set)} queries):")
print(f"  Average latency: {avg_time:.2f}s")
print(f"  Average context gain: {avg_gain:.1f}x")
print(f"\nKey advantages:")
print(f"  • Precise retrieval (small chunks)")
print(f"  • Rich context (parent chunks)")
print(f"  • {avg_gain:.1f}x more context than flat RAG")

## 🔟 Summary

### Hierarchical RAG Pattern

**Architecture:**
- Two-level structure: Parent (large) + Child (small) chunks
- Search precision: Match on small child chunks
- Generation richness: Use full parent context
- Automatic linking: Children know their parents

**Performance:**
- Context gain: 3-5x vs flat RAG
- Latency: +20-40% (parent fetching)
- Cost: +25-50% (larger context)
- Quality: Better for long documents

**When to Use:**
- ✅ Long documents (articles, papers)
- ✅ Need context preservation
- ✅ Structured content
- ❌ Short documents
- ❌ Cost-sensitive at scale

**Cost:** ~$0.10-0.12 per query (+25-50% vs Simple RAG)

---

**Pattern #22/37 Complete** ✅